# Aggregate model

This is the simplest DES model of our system, which can be run for the whole trust or per county - just change the CSV file of parameters provided.

## Imports

In [ ]:
import pandas as pd
import plotly.express as px

from rich import print

from ambdes import Model, Runner, SimConfig, ArrivalConfig, TimesConfig

## Data sources

Currently, the model uses two CSV files with synthetic parameters:

* `arrivals.csv` - mean number of arrivals by day of week and response category.
* `times.csv` - mean and standard deviation for different times, by response category.

In [ ]:
pd.read_csv("../data/arrivals.csv")

In [ ]:
pd.read_csv("../data/times.csv")

**TODO: ALL PARAMETERS FROM CSV - MOVE THE ONES CODED INTO SIMCONFIG INTO A CSV**

## Walk through the model

### Arrivals

The `ArrivalConfig` class loads `arrivals.csv` and uses it to define:

* **Patient arrivals:** a non-homogeneous Poisson process (NHPP) that varies by day of week. With a Poisson distribution, inter-arrival times are exponentially distributed.
* **Response categories:** assigned probabilistically based on observed proportions of each category.

In [ ]:
arrival_config = ArrivalConfig(arrival_csv="../data/arrivals.csv")

# Parameters used in NHPP
display(arrival_config.nspp_df)

display(arrival_config.category_proportions)

Our assumptions for arrivals are that:

* (a) Inter-arrival times vary by response category and day of week.
    * We agreed this in our discussion.
* (b) Proportion of each response category *do not* vary by day of week.
    * To check this assumption, we need to look at the proportion of C1 v.s., C2 v.s., C3 v.s., C4 by day of week...

In [ ]:
# Checking proportion of each response category by day of week
display(arrival_config.proportion_df)
display(arrival_config.variation_df)

### Times

The `TimesConfig` class loads `times.csv`. Currently, times are all modelled as lognormal distributions, so a mean and standard deviation (SD) is provided for each.

We have assumed that each of these times *vary by response category* - though we should check this in the real data - the model could be simplified to just one time across categories if it doesn't vary.

In [ ]:
times_config = TimesConfig(times_csv="../data/times.csv")
display(times_config.times_df)

### Configuration

The `SimConfig` class accepts instances of `ArrivalConfig` and `TimesConfig`, and creates a set of parameters ready for the model.

The distributions are all stored in `dist_config`, which follows a JSON format that can be accepted by the `sim-tools` `DistributionRegistry` class.

In [ ]:
config = SimConfig(
    arrival_config=arrival_config,
    times_config=times_config,
    warm_up_period=500,
    data_collection_period=10080,  # One week
)
print(config.__dict__)

### Model

The `Model` can be set-up by setting a run number and providing the `config` instance, then run by calling `run()`.

In [ ]:
model = Model(run_number=0, config=config)
model.run()

### Logger

We record a log using the `vidigi` `EventLogger` class. This means it can work with `vidigi` to produce animations or process flow charts if desired.

We also have the attributes each patient stored in the model - for example, here, we can look at the patient with ID one in `model.patients` and in the `log`.

In [ ]:
log = model.logger.to_dataframe()

# View patient with ID 1
print(model.patients[0].__dict__)
display(log[log["entity_id"] == 1])

### Runner

A `Runner` class is provided to run the model, calculate results from a run, and run the model for multiple replications.

In [ ]:
runner = Runner(config)
results = runner.run_reps()

## Results

This is an example of running the model for one replication, for one week, with no warm-up period.

### Mean response time and utilisation

We can view by run and overall.

In [ ]:
display(results["run"])

In [ ]:
display(results["overall"])

### Example plot: distribution of response times by response category

In [ ]:
df = pd.DataFrame(
    {
        "response_time": [p.response_time for p in model.patients],
        "category": [p.category for p in model.patients],
    }
)

fig = px.histogram(
    df,
    x="response_time",
    facet_col="category",
    category_orders={"category": ["C1", "C2", "C3", "C4"]},
    labels={
        "response_time": "Response time (minutes)",
        "category": "Category",
    },
    title="Distribution of response times by category",
)

fig.layout.yaxis.title.text = "Number of patients"

fig.show()